# 06. 파인튜닝 설정 하나 돌리기

`finetuning.ipynb`를 설정 하나(데이터 · 방식 · k · 학습률)로 실행하고, 결과 표와 그래프를 바로 보여준다. 여러 설정을 차례로 돌릴 때는 `06_finetune_queue.ipynb`.

- 코드는 `finetuning.ipynb` 하나뿐이다. 이 노트북은 설정 칸의 값만 바꿔서 실행한다
- 실행된 노트북 전체가 결과 폴더의 `executed.ipynb`에 남는다 (칸마다 출력 확인 가능)
- 같은 `TAG`로 이미 끝난 설정이면 건너뛴다 (`RERUN = True`면 결과 폴더를 지우고 다시)
- **A / B**: A = 윈도우로 나눈 뒤 같은 내용 윈도우 병합, B = 윈도우 전에 이벤트 반복 압축

**Colab에서**: GPU 런타임, 데이터는 `06_finetune_queue.ipynb`와 같은 방법으로 준비. 설정 하나는 GPU에서 5 ~ 15분.

## 1. 설정

In [ ]:
DATA = 'B'              # A(윈도우 병합) / B(이벤트 반복 압축)
MODE = 'both'           # tactic / technique / both (전술+기법: 학습은 함께, 추론·채점은 따로)
K = 16                  # 1, 3, 5, 8, 16
LR = '2e-5'
EPOCHS = 40
SUFFIX = '_fix'         # TAG = _lr{LR}_ep{EPOCHS}{SUFFIX}
SMOKE = False           # True: 2에폭·평가 일부 (몇 분)
RERUN = False           # True: 같은 TAG 결과를 지우고 다시

DATA_DIR = None         # None이면 자동 (06_finetune_queue와 같음)
FINETUNE_NB = None
WORK_DIR = None

VARIANT = f'{DATA}_{MODE}_k{K}'
TAG = f'_lr{LR}_ep{EPOCHS}{SUFFIX}'
print(VARIANT, TAG)

## 2. 준비

[볼 것] `분할 폴더 = 30 개`, GPU 이름.

In [ ]:
import os, sys, glob, subprocess, tarfile, importlib.util
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def find_data():
    """data/v2가 있는 폴더 (DATA_DIR). 없으면 드라이브의 ctgli_v2*.tar.xz를 /content/ctgli/data에 푼다."""
    if DATA_DIR and Path(DATA_DIR, 'v2').exists():
        return str(DATA_DIR)
    here = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    for c in [here / 'data', Path('/content/ctgli/data')]:
        if (c / 'v2').exists():
            return str(c)
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
        for z in sorted(glob.glob('/content/drive/MyDrive/**/ctgli_v2*.tar.xz', recursive=True)):
            os.makedirs('/content/ctgli/data', exist_ok=True)
            tarfile.open(z).extractall('/content/ctgli/data', filter='data'); print('압축 풀기:', z)
        if Path('/content/ctgli/data/v2').exists():
            return '/content/ctgli/data'
    raise FileNotFoundError('data/v2를 못 찾음: DATA_DIR을 지정하거나 드라이브에 ctgli_v2*.tar.xz를 올릴 것')

def find_notebook():
    if FINETUNE_NB and Path(FINETUNE_NB).exists():
        return str(FINETUNE_NB)
    for c in [Path.cwd() / 'finetuning.ipynb', Path.cwd() / 'notebooks' / 'finetuning.ipynb',
              Path('/content/ctgli/notebooks/finetuning.ipynb'), Path('/content/ctgli/finetuning.ipynb')]:
        if c.exists():
            return str(c)
    raise FileNotFoundError('finetuning.ipynb를 못 찾음: FINETUNE_NB를 지정할 것')

DATA_DIR, FINETUNE_NB = find_data(), find_notebook()
WORK_DIR = WORK_DIR or ('/content/work' if IN_COLAB else os.path.join(DATA_DIR, 'finetune_work'))
need = [p for p, m in [('nbclient', 'nbclient'), ('nbformat', 'nbformat'), ('ipykernel', 'ipykernel')] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)
gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip()
print('DATA_DIR    =', DATA_DIR)
print('FINETUNE_NB =', FINETUNE_NB)
print('WORK_DIR    =', WORK_DIR)
print('GPU         =', gpu or '없음 (CPU로는 매우 느림)')
print('분할 폴더    =', len(glob.glob(f'{DATA_DIR}/v2/[AB]_*_k*/splits')), '개')

## 3. 실행

[하는 일] `finetuning.ipynb`를 이 설정으로 끝까지 실행한다 (칸이 끝날 때까지 기다림). 학습 중 진행은 출력되지 않고, 끝나면 실행 줄(`k= ... seed= ...`)이 나온다.

In [ ]:
RUNNER = r"""
import json, os, re, sys, time
import nbformat
from nbclient import NotebookClient

cfg = json.loads(sys.argv[1])
os.environ['DATA_DIR'], os.environ['WORK_DIR'] = cfg['data_dir'], cfg['work_dir']

def params(nb, variant, lr, epochs, tag, smoke):
    # finetuning.ipynb 설정 칸의 원문을 바꾼다. 원문이 달라졌으면 멈춘다 (다른 설정으로 조용히 도는 것을 막음)
    reps = [("VARIANT = 'B_tactic_k16'", f"VARIANT = '{variant}'"),
            ("EPOCHS = 10", f"EPOCHS = {epochs}"),
            ("TAG = ''", f"TAG = '{tag}'")]
    if smoke:
        reps.append(("SMOKE = False", "SMOKE = True"))
    cell = next(c for c in nb.cells if c.cell_type == 'code' and "VARIANT = 'B_tactic_k16'" in c.source)
    for a, b in reps:
        assert a in cell.source, f'finetuning.ipynb 설정 칸에 {a!r}가 없음'
        cell.source = cell.source.replace(a, b)
    cell.source = re.sub(r"ENCODER_LR = [0-9e.-]+", f"ENCODER_LR = {lr}", cell.source)
    cell.source = re.sub(r"OTHERS_LR = [0-9e.-]+", f"OTHERS_LR = {lr}", cell.source)

for variant, lr in cfg['jobs']:
    tag = f"_lr{lr}_ep{cfg['epochs']}{cfg['suffix']}"
    out = f"{cfg['data_dir']}/v2/{variant}/results/finetune_{variant}{tag}" + ('_smoke' if cfg['smoke'] else '')
    if os.path.exists(f'{out}/executed.ipynb'):
        print(variant, lr, 'skip (done)', flush=True); continue
    nb = nbformat.read(cfg['notebook'], 4)
    params(nb, variant, lr, cfg['epochs'], tag, cfg['smoke'])
    t = time.time()
    try:
        NotebookClient(nb, kernel_name='python3', timeout=6 * 3600,
                       resources={'metadata': {'path': os.path.dirname(cfg['notebook'])}}).execute()
        status = 'done'
    except Exception as e:
        status = 'ERROR ' + (str(e).strip().splitlines() or [''])[-1][:300]
    os.makedirs(out, exist_ok=True)
    nbformat.write(nb, f'{out}/executed.ipynb' if status == 'done' else f'{out}/failed.ipynb')
    for c in nb.cells:
        if c.cell_type == 'code':
            for o in c.outputs:
                for line in (o.get('text') or '').splitlines():
                    if line.startswith(('k=', '검사', '  학습', '  추론')):
                        print(variant, lr, line, flush=True)
    print(variant, lr, status, round(time.time() - t), 's', flush=True)
print('ALL FINISHED', flush=True)
"""
RUNNER_PY = os.path.join(WORK_DIR, 'finetune_runner.py')
os.makedirs(WORK_DIR, exist_ok=True)
open(RUNNER_PY, 'w').write(RUNNER)
import json, shutil
OUT = f'{DATA_DIR}/v2/{VARIANT}/results/finetune_{VARIANT}{TAG}' + ('_smoke' if SMOKE else '')
if RERUN and os.path.exists(OUT):
    shutil.rmtree(OUT); print('지움:', OUT)
cfg = json.dumps(dict(jobs=[(VARIANT, LR)], epochs=EPOCHS, suffix=SUFFIX, smoke=SMOKE, data_dir=DATA_DIR,
                      work_dir=WORK_DIR, notebook=os.path.abspath(FINETUNE_NB)))
r = subprocess.run([sys.executable, RUNNER_PY, cfg], capture_output=True, text=True)
print('\n'.join(l for l in r.stdout.splitlines() if not l.startswith(('0.00s', '  -'))))
if r.returncode != 0:
    print(r.stderr[-3000:])
print('결과 폴더:', OUT)

## 4. 결과

[하는 일] 이 설정의 `runs.csv`(k = 0은 기준선)와 그래프를 보여준다. 다른 설정과 비교는 `06_finetune_queue.ipynb`의 5번 칸.

In [ ]:
import pandas as pd
from IPython.display import Image, display
runs = pd.read_csv(f'{OUT}/runs.csv')
cols = ['k', 'seed', 'best_epoch'] + [c for c in runs.columns if c.endswith(('seen_macro_f1', 'held_macro_ap', 'heldk_macro_ap', 'seen_separation'))] + ['agnews_acc']
display(runs[cols].round(3))
print('seed 평균 (k > 0):')
display(runs[runs.k != 0][cols[3:]].mean().round(3).to_frame('mean').T)
for f in ['fig_kshot.png', 'fig_epochs.png', 'fig_per_label.png']:
    if os.path.exists(f'{OUT}/{f}'):
        display(Image(f'{OUT}/{f}', width=800))